In [1]:
import pandas as pd
from google import genai
import os

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY") 
client = genai.Client(api_key=GOOGLE_API_KEY)

In [2]:
from google.genai import types

event_schema = types.Schema(
    type=types.Type.OBJECT,
    properties={
        "event_summary": types.Schema(type=types.Type.STRING),
        "key_actors": types.Schema(type=types.Type.ARRAY, items=types.Schema(type=types.Type.STRING)),
        "event_time": types.Schema(type=types.Type.STRING, format="date"),
        "fetch_success": types.Schema(type=types.Type.BOOLEAN),
    },
    required=["event_summary", "key_actors", "event_time", "fetch_success"]
)

gdelt_event_df = pd.read_csv("gdelt_event.csv")

In [3]:
import json

google_search_tool = types.Tool(google_search=types.GoogleSearch())

def get_event_summary(row):
    url = row["SOURCEURL"]

    prompt = prompt = f"""You are a senior news reporter specializing in geopolitical and economic events.
        Your task is to identify the key news event from the provided data and write a concise headline summary.

        Follow these steps in order, stopping as soon as you successfully identify the event:
        1. **Fetch URL**: Attempt to retrieve and read the article at: {url}
        2. **Search**: If the URL is inaccessible, use google_search to find news matching the location, actors, and date.
        3. **Knowledge**: If search yields nothing, draw from your own knowledge of events matching this context.

        Context:
        - Date: {row['SQLDATE']}
        - Location: {row['ActionGeo_FullName']}
        - Actor 1: {row['Actor1Name']} ({row['Actor1Type1Code']}, {row['Actor1CountryCode']})
        - Actor 2: {row['Actor2Name']} ({row['Actor2Type1Code']}, {row['Actor2CountryCode']})
        - Event Code: {row['EventCode']} (Root: {row['EventRootCode']})
        - Goldstein Scale: {row['GoldsteinScale']} (negative = conflict, positive = cooperation)
        - Avg Media Tone: {row['AvgTone']}
        - Number of Articles: {row['NumArticles']}

        Writing guidelines for event_summary:
        - 1 to 3 sentences maximum
        - Lead with the most newsworthy fact (who did what, where, when)
        - Use active voice and precise language
        - No preamble like "The most impactful event..." — just report it directly

        Set fetch_success=true only if you successfully retrieved content from the URL.
        Set source to "url", "web_search", or "knowledge" depending on which step succeeded."""

    response = client.models.generate_content(
        model="gemini-3-flash-preview",
        contents=prompt,
        config=types.GenerateContentConfig(
            tools=[google_search_tool],
            response_mime_type="application/json",
            response_schema=event_schema,
        )
    )

    return pd.Series(json.loads(response.text))

In [4]:
import time

results = []
total = len(gdelt_event_df)

for i, (idx, row) in enumerate(gdelt_event_df.iterrows()):
    print(f"Processing {i+1}/{total} (GLOBALEVENTID={row.get('GLOBALEVENTID', idx)})...", end=" ")
    try:
        summary = get_event_summary(row)
        summary["GLOBALEVENTID"] = row.get("GLOBALEVENTID", idx)
        results.append(summary)
        print("OK")
    except Exception as e:
        print(f"FAILED — {e}")
        results.append(pd.Series({
            "GLOBALEVENTID": row.get("GLOBALEVENTID", idx),
            "event_summary": None,
            "key_actors": None,
            "event_time": None,
            "fetch_success": False,
        }))
    # Respect rate limits
    time.sleep(1)

summary_df = pd.DataFrame(results)

# Merge back with original event data
result_df = gdelt_event_df.merge(summary_df, on="GLOBALEVENTID", how="left")

print(f"\nDone. {summary_df['event_summary'].notna().sum()}/{total} events summarized successfully.")
result_df.head()

Processing 1/2400 (GLOBALEVENTID=435199599)... OK
Processing 2/2400 (GLOBALEVENTID=452406158)... OK
Processing 3/2400 (GLOBALEVENTID=491126088)... OK
Processing 4/2400 (GLOBALEVENTID=423818581)... OK
Processing 5/2400 (GLOBALEVENTID=436606953)... OK
Processing 6/2400 (GLOBALEVENTID=473766932)... OK
Processing 7/2400 (GLOBALEVENTID=431842051)... OK
Processing 8/2400 (GLOBALEVENTID=556115558)... OK
Processing 9/2400 (GLOBALEVENTID=434749057)... OK
Processing 10/2400 (GLOBALEVENTID=441193793)... OK
Processing 11/2400 (GLOBALEVENTID=441193790)... OK
Processing 12/2400 (GLOBALEVENTID=453904113)... OK
Processing 13/2400 (GLOBALEVENTID=465809250)... OK
Processing 14/2400 (GLOBALEVENTID=484369020)... OK
Processing 15/2400 (GLOBALEVENTID=484369019)... OK
Processing 16/2400 (GLOBALEVENTID=447072731)... OK
Processing 17/2400 (GLOBALEVENTID=490052672)... OK
Processing 18/2400 (GLOBALEVENTID=457712901)... OK
Processing 19/2400 (GLOBALEVENTID=437898540)... OK
Processing 20/2400 (GLOBALEVENTID=489256

,event_year,rank_in_year,GLOBALEVENTID,SQLDATE,Actor1Name,Actor1Type1Code,Actor1CountryCode,Actor2Name,Actor2Type1Code,Actor2CountryCode,...,ActionGeo_CountryCode,GoldsteinScale,NumArticles,AvgTone,impact_score,SOURCEURL,event_summary,key_actors,event_time,fetch_success
0,2015,1,435199599,20150522,IRAQI,GOV,IRQ,SHIA,UAF,NaN,...,IZ,-7.2,6,-53.962901,19.652643,http://freerepublic.com/focus/f-news/3292340/p...,Thousands of Shia militia fighters arrived at ...,"[IRAQI GOVERNMENT, SHIA MILITIAS, ISIS]",2015-05-22,False
1,2015,2,452406158,20150726,NIGERIA,NaN,NGA,NaN,NaN,NaN,...,NI,-10.0,20,-44.444444,18.246690,https://www.blesk.cz/clanek/zpravy-svet/332959...,A female suicide bomber detonated an explosive...,"[Nigeria, Boko Haram]",2015-07-26,False
2,2015,3,491126088,20151204,CZECH,NaN,CZE,NaN,NaN,NaN,...,EZ,-10.0,6,-36.363636,15.492864,http://www.blesk.cz/diskuse/488003,Czech Member of the European Parliament Milosl...,"[Miloslav Ransdorf, Swiss Police, Zürcher Kant...",2015-12-04,False
3,2015,4,423818581,20150408,BOSTON,NaN,USA,NaN,NaN,NaN,...,US,-10.0,10,-33.333333,14.719369,http://www.blesk.cz/diskuse/411435,A federal jury in Boston convicted Dzhokhar Ts...,"[Dzhokhar Tsarnaev, Federal Jury]",2015-04-08,False
4,2015,5,436606953,20150527,SINAI,NaN,EGY,POLICEMAN,COP,NaN,...,NaN,-10.0,20,-30.000000,13.913357,https://www.blesk.cz/clanek/zpravy-live-zpravy...,An Egyptian policeman was killed and eight oth...,"[Egyptian Policeman, Sinai Province, Egyptian ...",2015-05-27,False


In [5]:
result_df.to_csv("gdelt_events_with_summaries.csv", index=False)
print(f"Exported {len(result_df)} rows to gdelt_events_with_summaries.csv")

Exported 2400 rows to gdelt_events_with_summaries.csv
